# Smoke Test Iceberg REST Catalog with PyIceberg & Polars

Notebook này thực hiện smoke test kết nối đến Iceberg REST Catalog và tích hợp với Polars, bao gồm:
1. Kết nối với Iceberg REST Catalog đang chạy trên `http://localhost:8181`.
2. Tạo namespace `default` nếu chưa tồn tại.
3. Tạo bảng Iceberg mới `default.smoke_test_table`.
4. Ghi dữ liệu ban đầu vào bảng Iceberg bằng PyArrow.
5. Đọc dữ liệu từ Iceberg bằng **Polars** (`pl.scan_iceberg`).
6. Append thêm dữ liệu bằng Polars DataFrame.
7. Thực hiện truy vấn **Time Travel** (theo Snapshot ID và Timestamp).

In [1]:
import os
import time
import polars as pl
import pyarrow as pa
from pyiceberg.catalog import load_catalog
from dotenv import load_dotenv

# Load environment variables từ file .env
load_dotenv(dotenv_path="../.env")

True

## 1. Khởi tạo Catalog Client

In [2]:
CATALOG_URI = os.getenv("CATALOG_URI_CLIENT", "http://localhost:8181")
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://localhost:9000")
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID", "admin")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "admin123")

catalog = load_catalog(
    "demo",
    **{
        "type": "rest",
        "uri": CATALOG_URI,
        "s3.endpoint": MINIO_ENDPOINT,
        "s3.access-key-id": AWS_ACCESS_KEY_ID,
        "s3.secret-access-key": AWS_SECRET_ACCESS_KEY,
        "s3.path-style-access": "true"
    }
)
print("Catalog client initialized successfully!")

Catalog client initialized successfully!


## 2. Tạo Namespace

In [3]:
try:
    catalog.create_namespace("default")
    print("Namespace 'default' created successfully.")
except Exception as e:
    print("Namespace 'default' already exists or other state:", e)

Namespace 'default' already exists or other state: AlreadyExistsException: Namespace already exists: default


## 3. Tạo bảng Iceberg

In [4]:
table_name = "default.smoke_test_table"

# Xóa bảng nếu đã tồn tại để chạy lại sạch sẽ
try:
    catalog.drop_table(table_name)
    print(f"Dropped existing table {table_name}.")
except Exception:
    pass

# Định nghĩa Schema cho bảng Iceberg
schema = pa.schema([
    ("id", pa.int64()),
    ("name", pa.string()),
    ("score", pa.float64())
])

table = catalog.create_table(table_name, schema=schema)
print(f"Table '{table_name}' created successfully.")

Dropped existing table default.smoke_test_table.
Table 'default.smoke_test_table' created successfully.


## 4. Append dữ liệu đợt 1 (Tạo Snapshot đầu tiên)

In [5]:
data1 = pa.table({
    "id": [1, 2, 3],
    "name": ["Vietnam", "USA", "Singapore"],
    "score": [95.5, 90.0, 98.2]
})

print("Appending first batch of data...")
table.append(data1)
print("First batch appended successfully!")

# Ghi nhận thời gian sau khi ghi đợt 1 để test time travel theo timestamp
time.sleep(1)  # Chờ 1 chút để phân biệt timestamp rõ ràng
timestamp_after_batch1 = int(time.time() * 1000)
print(f"Timestamp after batch 1: {timestamp_after_batch1}")

Appending first batch of data...
First batch appended successfully!
Timestamp after batch 1: 1781423310660


## 5. Đọc dữ liệu bằng Polars
Để Polars có thể scan Iceberg từ MinIO, chúng ta cần cung cấp `storage_options` chứa thông tin endpoint của MinIO.

In [6]:
storage_options = {
    "aws_access_key_id": AWS_ACCESS_KEY_ID,
    "aws_secret_access_key": AWS_SECRET_ACCESS_KEY,
    "aws_endpoint_url": MINIO_ENDPOINT,
    "aws_allow_http": "true",
    "aws_region": "us-east-1"
}

# Reload table object từ catalog để cập nhật trạng thái mới nhất
table = catalog.load_table(table_name)

print("Scanning table with Polars...")
lf = pl.scan_iceberg(table, storage_options=storage_options)
df = lf.collect()
print("Data read by Polars:")
print(df)

Scanning table with Polars...
Data read by Polars:
shape: (3, 3)
┌─────┬───────────┬───────┐
│ id  ┆ name      ┆ score │
│ --- ┆ ---       ┆ ---   │
│ i64 ┆ str       ┆ f64   │
╞═════╪═══════════╪═══════╡
│ 1   ┆ Vietnam   ┆ 95.5  │
│ 2   ┆ USA       ┆ 90.0  │
│ 3   ┆ Singapore ┆ 98.2  │
└─────┴───────────┴───────┘


## 6. Append dữ liệu đợt 2 bằng Polars DataFrame (Tạo Snapshot thứ hai)
Chúng ta sẽ chuẩn bị một Polars DataFrame, convert nó sang PyArrow Table rồi append vào Iceberg table.

In [7]:
# Tạo DataFrame bằng Polars
df_new = pl.DataFrame({
    "id": [4, 5],
    "name": ["Japan", "Germany"],
    "score": [88.5, 92.0]
})

print("Converting Polars DataFrame to PyArrow and appending...")
table.append(df_new.to_arrow())
print("Second batch appended successfully!")

# Load lại table để cập nhật danh sách snapshots mới nhất
table = catalog.load_table(table_name)

# Đọc lại toàn bộ bảng bằng Polars để kiểm tra kết quả hiện tại
print("\nScanning current state with Polars:")
df_all = pl.scan_iceberg(table, storage_options=storage_options).collect()
print(df_all)

Converting Polars DataFrame to PyArrow and appending...
Second batch appended successfully!

Scanning current state with Polars:
shape: (5, 3)
┌─────┬───────────┬───────┐
│ id  ┆ name      ┆ score │
│ --- ┆ ---       ┆ ---   │
│ i64 ┆ str       ┆ f64   │
╞═════╪═══════════╪═══════╡
│ 4   ┆ Japan     ┆ 88.5  │
│ 5   ┆ Germany   ┆ 92.0  │
│ 1   ┆ Vietnam   ┆ 95.5  │
│ 2   ┆ USA       ┆ 90.0  │
│ 3   ┆ Singapore ┆ 98.2  │
└─────┴───────────┴───────┘


## 7. Kiểm nghiệm Time Travel
Iceberg lưu trữ lịch sử thay đổi thông qua các Snapshots. Chúng ta có thể quay ngược thời gian để xem dữ liệu ở các snapshot trước đó.

In [8]:
print("Lịch sử Snapshots của bảng:")
snapshots = list(table.snapshots())
for idx, s in enumerate(snapshots):
    print(f"Snapshot {idx+1}: ID={s.snapshot_id}, Created at={s.timestamp_ms}")

# Lấy Snapshot ID đầu tiên (chỉ chứa 3 dòng dữ liệu)
first_snapshot_id = snapshots[0].snapshot_id
print(f"\nTargeting Snapshot ID: {first_snapshot_id} (Batch 1)")

Lịch sử Snapshots của bảng:
Snapshot 1: ID=4462335856742110241, Created at=1781423309584
Snapshot 2: ID=8056683312863092071, Created at=1781423315334

Targeting Snapshot ID: 4462335856742110241 (Batch 1)


### A. Time Travel bằng Snapshot ID
Sử dụng tham số `snapshot_id` trong PyIceberg `scan()` để lấy dữ liệu tương ứng của snapshot đầu tiên.

In [9]:
# Thực hiện scan với snapshot_id xác định
scan_snap1 = table.scan(snapshot_id=first_snapshot_id)
df_snap1 = pl.from_arrow(scan_snap1.to_arrow())

print(f"Data from Snapshot {first_snapshot_id} (should only have 3 rows):")
print(df_snap1)

print("Success: Time travel via Snapshot ID works!")

Data from Snapshot 4462335856742110241 (should only have 3 rows):
shape: (3, 3)
┌─────┬───────────┬───────┐
│ id  ┆ name      ┆ score │
│ --- ┆ ---       ┆ ---   │
│ i64 ┆ str       ┆ f64   │
╞═════╪═══════════╪═══════╡
│ 1   ┆ Vietnam   ┆ 95.5  │
│ 2   ┆ USA       ┆ 90.0  │
│ 3   ┆ Singapore ┆ 98.2  │
└─────┴───────────┴───────┘
Success: Time travel via Snapshot ID works!


### B. Time Travel bằng Timestamp
Chúng ta có thể tìm Snapshot tương ứng với một mốc thời gian cụ thể (ví dụ mốc `timestamp_after_batch1` đã lưu trước đó).

In [10]:
print(f"Looking for data as of timestamp: {timestamp_after_batch1}")

# Tìm snapshot phù hợp với mốc thời gian bằng snapshot_as_of_timestamp
target_snapshot = table.snapshot_as_of_timestamp(timestamp_after_batch1)

if target_snapshot is not None:
    print(f"Found matching Snapshot ID: {target_snapshot.snapshot_id} committed at {target_snapshot.timestamp_ms}")
    
    scan_ts = table.scan(snapshot_id=target_snapshot.snapshot_id)
    df_ts = pl.from_arrow(scan_ts.to_arrow())
    
    print("\nData from Time Travel by Timestamp (should only have 3 rows):")
    print(df_ts)

    print("SUCCESS: Iceberg + Polars smoke test with Time Travel completed successfully!")
else:
    print("No snapshot found at or before the given timestamp.")

Looking for data as of timestamp: 1781423310660
Found matching Snapshot ID: 4462335856742110241 committed at 1781423309584

Data from Time Travel by Timestamp (should only have 3 rows):
shape: (3, 3)
┌─────┬───────────┬───────┐
│ id  ┆ name      ┆ score │
│ --- ┆ ---       ┆ ---   │
│ i64 ┆ str       ┆ f64   │
╞═════╪═══════════╪═══════╡
│ 1   ┆ Vietnam   ┆ 95.5  │
│ 2   ┆ USA       ┆ 90.0  │
│ 3   ┆ Singapore ┆ 98.2  │
└─────┴───────────┴───────┘
SUCCESS: Iceberg + Polars smoke test with Time Travel completed successfully!
